In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("10-udfs")
dfs = register_views(spark)
emp = dfs["employees"]

# ── Section 1: Simple Python UDF (registered) ────────────────────────────────
# UDFs are plain Python functions wrapped for Spark.
# Return type must be declared; Spark uses it to deserialize the result.
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def salary_band_udf(salary):
    if salary is None: return "Unknown"   # emp 10, 15 → NULL salary
    if salary == 0.0:  return "Zero/Flaw" # emp 19 → salary = 0.0
    if salary < 80000: return "Junior"
    if salary < 130000: return "Mid"
    if salary < 180000: return "Senior"
    return "Lead"

print("=== Section 1: salary_band_udf groupBy count ===")
emp.withColumn("band", salary_band_udf(F.col("salary"))) \
   .groupBy("band").count().show()

# ── Section 2: Register UDF for Spark SQL ─────────────────────────────────────
# spark.udf.register makes a UDF callable from SQL strings.
# The registered name ("salary_band_sql") is used in SQL; the function object
# carries the return type already defined above.
spark.udf.register("salary_band_sql", salary_band_udf)

print("=== Section 2: salary_band_sql via spark.sql ===")
spark.sql("SELECT salary_band_sql(salary) AS band, COUNT(*) FROM employees GROUP BY 1").show()

# ── Section 3: UDF with multiple input columns ────────────────────────────────
# UDFs can accept multiple columns; each maps to a positional argument.
@udf(returnType=StringType())
def full_name_udf(first, last):
    if first is None or last is None: return None
    return f"{first} {last}"

print("=== Section 3: full_name_udf (multi-column) ===")
emp.withColumn("full_name", full_name_udf(F.col("first_name"), F.col("last_name"))) \
   .select("emp_id", "full_name").show(5)

# ── Section 4: UDF performance warning and native function alternative ────────
# UDF: each row is serialized Python ↔ JVM ↔ Python (slow, breaks optimizer).
# Native (F.when / built-ins): stays in JVM, benefits from Catalyst & Tungsten.
# Rule: always prefer native functions when a built-in equivalent exists.

print("=== Section 4a: band via UDF (slow path) ===")
emp.withColumn("band_udf", salary_band_udf(F.col("salary"))).select("emp_id", "band_udf").show(3)

print("=== Section 4b: band via native F.when (fast path) ===")
emp.withColumn("band_native",
    F.when(F.col("salary").isNull(), "Unknown")
     .when(F.col("salary") == 0, "Zero/Flaw")
     .when(F.col("salary") < 80000, "Junior")
     .when(F.col("salary") < 130000, "Mid")
     .when(F.col("salary") < 180000, "Senior")
     .otherwise("Lead")
).select("emp_id", "band_native").show(3)
# Comment: native is always preferred when a built-in equivalent exists

# ── Section 5: Pandas UDF (vectorized UDF, much faster than row-by-row) ───────
# pandas_udf operates on batches (pd.Series), not individual rows.
# Drastically less Python-JVM serialization overhead vs. row-by-row UDF.
# Requires: pyarrow installed.
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(DoubleType())
def bonus_pandas_udf(salary: pd.Series) -> pd.Series:
    return (salary.fillna(0) * 0.10).round(2)

print("=== Section 5: bonus_pandas_udf (vectorized) ===")
emp.withColumn("bonus", bonus_pandas_udf(F.col("salary"))) \
   .select("emp_id", "salary", "bonus").show(5)
# Comment: pandas_udf operates on batches (Series), not individual rows
# → far less Python-JVM serialization overhead

# ── Section 6: Pandas UDF with multiple input Series ─────────────────────────
# Multiple columns map to multiple pd.Series arguments, processed batch-wise.
@pandas_udf(StringType())
def full_name_pandas(first: pd.Series, last: pd.Series) -> pd.Series:
    return first + " " + last

print("=== Section 6: full_name_pandas (multi-Series pandas_udf) ===")
emp.withColumn("full_name", full_name_pandas(F.col("first_name"), F.col("last_name"))) \
   .select("emp_id", "full_name").show(5)

# ── Section 7: UDF that returns a struct (complex type) ──────────────────────
# Return a dict matching the StructType schema; Spark maps dict keys → fields.
# Access sub-fields with dot notation: col("info.band"), col("info.bonus").
from pyspark.sql.types import StructType, StructField

result_schema = StructType([
    StructField("band",  StringType()),
    StructField("bonus", DoubleType()),
])

@udf(returnType=result_schema)
def salary_info_udf(salary):
    if salary is None: return {"band": "Unknown", "bonus": 0.0}
    band = "Lead" if salary > 180000 else "Senior" if salary > 130000 else "Mid"
    return {"band": band, "bonus": round(salary * 0.10, 2)}

print("=== Section 7: salary_info_udf (struct return type) ===")
result = emp.withColumn("info", salary_info_udf(F.col("salary")))
result.select("emp_id", "salary", "info.band", "info.bonus").show(5)

# ── Section 8: UDF caveats – NULL handling and exception safety ───────────────
# UDFs do NOT propagate Python exceptions cleanly to Spark error messages.
# Always handle None inside the UDF body; never assume non-null input.
# NULL in → NULL out is standard behavior UNLESS you guard against it.

@udf(returnType=StringType())
def safe_email_domain(email):
    # emp 22 has NULL email; without this guard the split would raise AttributeError
    if email is None: return "no-domain"
    return email.split("@")[-1] if "@" in email else "invalid"

print("=== Section 8: safe_email_domain (NULL-safe UDF) ===")
emp.withColumn("domain", safe_email_domain(F.col("email"))) \
   .select("emp_id", "email", "domain").show(5)
# emp 22 (NULL email) will show "no-domain" instead of crashing

stop_and_wait(spark)